In [2]:
import os
import base64
from dotenv import load_dotenv

from openai import OpenAI
from agents import Agent, Runner, function_tool


# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is Missing")


# OpenAI client
openai_client = OpenAI()


# ---------------------------------------------------------
# Image Generation Tool
# ---------------------------------------------------------

@function_tool
def generate_image(prompt: str) -> str:
    """Generate an image using OpenAI's image API."""

    print(f"🎨 Generating image for: {prompt}")

    response = openai_client.images.generate(
        model="gpt-image-2",
        prompt=prompt,
        size="1024x1024",
        quality="high",
        n=1
    )

    # GPT Image returns Base64 image data
    image_base64 = response.data[0].b64_json

    # Decode Base64
    image_bytes = base64.b64decode(image_base64)

    # Save image locally
    image_path = "generated_image.png"

    with open(image_path, "wb") as file:
        file.write(image_bytes)

    print(f"✅ generate_image: done")
    print(f"✅ Image saved to: {image_path}")

    return image_path


# ---------------------------------------------------------
# Agent
# ---------------------------------------------------------

IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed gpt-image-2 prompt for a hero image.

Rules for your gpt-image-2 prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
"""

agent = Agent(
    name="Image Agent",

    model="gpt-4o-mini",

    instructions= IMAGE_AGENT_PROMPT,

    tools=[generate_image]
)


# ---------------------------------------------------------
# Run Agent
# ---------------------------------------------------------

result = await Runner.run(
    agent,
    "Create a photorealistic image of San Francisco skyline at sunset."
)


print("\nFinal response:")
print(result.final_output)

🎨 Generating image for: A stunning, photorealistic view of the San Francisco skyline during sunset. The image captures the iconic Golden Gate Bridge gracefully arching over the bay, silhouetted against a vibrant sky filled with hues of orange, pink, and purple. The sun is setting on the horizon, casting a warm golden light that reflects off the glass buildings of downtown San Francisco. In the foreground, the gently rippling waters of the bay mirror the colorful sky, creating a serene and captivating atmosphere. The scene is devoid of people, allowing the viewer to focus on the breathtaking beauty of the skyline and the natural landscape. Soft, wispy clouds enhance the dramatic sunset effect, adding depth and richness to the overall mood.
✅ generate_image: done
✅ Image saved to: generated_image.png

Final response:
Here is the photorealistic image of the San Francisco skyline at sunset. Let me know if you need anything else!
